## Nifty 50 Options Trading Strategy Bot

This script implements a Nifty 50 options trading strategy with dynamic profit locking, re-entry rules, and various safety features. It uses Zerodha Kite Connect for trading and market data.


### 1. Configuration

Adjust the following parameters to customize the strategy and trading environment.

In [ ]:
import logging

# Install Kite Connect library
!pip install kiteconnect --quiet

In [ ]:
import datetime
import time
import math
import logging
import pandas as pd
from kiteconnect import KiteConnect
from collections import defaultdict

# --- General Configuration ---
PRODUCT_TYPE = "MIS"  # MIS (Intraday), NRML (Carry Forward)
ORDER_TYPE = "MARKET" # MARKET, LIMIT, SL, SL-M
QUANTITY_PER_LOT = 50 # Nifty lot size
NIFTY_FUTURES_INSTRUMENT_TOKEN = 256265 # Example for Nifty 50 futures for reference, if direct spot is not available easily

# --- Strategy Parameters ---
FIRST_CANDLE_START_TIME = datetime.time(9, 15) # Strategy start time
FIRST_CANDLE_END_TIME = datetime.time(9, 18)   # End of first 3-min candle
PROFIT_ENTRY_THRESHOLD_PERCENT = 0.07          # Nifty spot change to trigger trade (+0.07% or -0.07%)
OPTION_STRIKE_DIFFERENCE = 50                  # Difference for OTM/ITM strikes (e.g., 50 points)
MARKET_CLOSE_TIME = datetime.time(15, 15)      # Auto square-off time

# --- Dynamic Profit Lock (Trailing Profit Lock) ---
DYNAMIC_PROFIT_LOCK_ENABLED = True             # Enable/Disable dynamic profit lock
INITIAL_PROFIT_LOCK_TRIGGER = 600              # Initial P&L (₹) to activate profit lock
PROFIT_LOCK_BUFFER = 100                       # Buffer (₹) between current P&L and locked P&L

# --- Risk Management ---
DAILY_MAX_LOSS = 1000                          # Daily maximum loss (₹). If reached, stop trading.

# --- Trading Modes ---
PAPER_TRADING_MODE = True                      # Set to True for paper trading, False for live trading

# --- Kite Connect API Configuration ---
# Store these securely in Colab's secrets manager
# For example: KITE_API_KEY, KITE_API_SECRET, KITE_REQUEST_TOKEN
# Access them using:
# from google.colab import userdata
# KITE_API_KEY = userdata.get('KITE_API_KEY')
# KITE_API_SECRET = userdata.get('KITE_API_SECRET')

# Request token needs to be generated daily by logging into Zerodha Kite
# and redirecting to your app's redirect_url. It's usually a URL parameter.
# For paper trading, you might mock this or use a long-lived token if available.

# --- Logging Configuration ---
LOG_FILE = "nifty_algo_trade.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("Configuration loaded successfully.")


### 2. Kite Connect Initialization and Helper Functions

This section handles the connection to Zerodha Kite and provides utility functions for API interactions.

In [ ]:
# Initialize KiteConnect
def initialize_kite_api(api_key, api_secret, request_token=None):
    try:
        kite = KiteConnect(api_key=api_key)
        if request_token:
            # Generate access token if a request token is provided
            data = kite.generate_session(request_token, api_secret=api_secret)
            kite.set_access_token(data["access_token"])
            logger.info("Kite Connect session generated and access token set.")
        else:
            logger.warning("No request token provided. If starting fresh, you need to generate one.")
            logger.info("Manual login URL: %s", kite.login_url())
        return kite
    except Exception as e:
        logger.error(f"Error initializing Kite Connect: {e}")
        return None

# Placeholder for global Kite object
kite = None

# --- Helper function to reconnect Kite (if needed) ---
def reconnect_kite(api_key, api_secret, request_token):
    global kite
    logger.info("Attempting to reconnect Kite...")
    kite = initialize_kite_api(api_key, api_secret, request_token)
    if kite:
        logger.info("Kite reconnected successfully.")
    else:
        logger.error("Failed to reconnect Kite. Check credentials and network.")
    return kite


# --- Order Placement and Management ---

# Placeholder for executed orders to prevent duplicates and track positions
# Format: {order_id: {'instrument_token': token, 'tradingsymbol': symbol, 'quantity': qty, 'type': 'BUY'/'SELL'}}
EXECUTED_ORDERS = {}

# Track active positions (for P&L calculation and square-off)
# Format: {tradingsymbol: {'quantity': net_qty, 'entry_price': avg_entry_price, 'order_ids': [...]}}
ACTIVE_POSITIONS = defaultdict(lambda: {'quantity': 0, 'entry_price': 0, 'order_ids': []})

# Global P&L variables
NET_REALIZED_PNL = 0
NET_UNREALIZED_PNL = 0

def calculate_realized_pnl(order_details):
    global NET_REALIZED_PNL
    # In a real scenario, this would be fetched from Kite's order/trade book
    # For simulation, we'll assume a simplified calculation for now
    logger.info(f"Calculating realized P&L for: {order_details}")
    # Add logic to calculate realized P&L based on executed orders
    # This is a complex part often handled by fetching actual trade book from broker

def place_order(instrument_token, tradingsymbol, transaction_type, quantity, order_type=ORDER_TYPE, product=PRODUCT_TYPE, price=0):
    global kite
    if PAPER_TRADING_MODE:
        logger.info(f"[PAPER TRADE] Placing {transaction_type} order for {tradingsymbol} quantity {quantity}")
        # Simulate order ID for paper trading
        order_id = f"PAPER_ORDER_{int(time.time() * 1000)}"
        # Simulate successful execution
        EXECUTED_ORDERS[order_id] = {
            'instrument_token': instrument_token,
            'tradingsymbol': tradingsymbol,
            'quantity': quantity,
            'type': transaction_type,
            'status': 'complete',
            'average_price': price if price > 0 else get_ltp(instrument_token) # Use price if limit, else LTP
        }
        update_active_positions(tradingsymbol, transaction_type, quantity, EXECUTED_ORDERS[order_id]['average_price'])
        logger.info(f"[PAPER TRADE] Order simulated. Order ID: {order_id}")
        return order_id

    if not kite:
        logger.error("Kite object not initialized. Cannot place order.")
        return None

    try:
        order_params = {
            "variety": kite.VARIETY_REGULAR,
            "exchange": kite.EXCHANGE_NFO, # Nifty Futures & Options
            "tradingsymbol": tradingsymbol,
            "transaction_type": transaction_type, # kite.TRANSACTION_TYPE_BUY or kite.TRANSACTION_TYPE_SELL
            "quantity": quantity,
            "product": product,
            "order_type": order_type
        }
        if order_type == kite.ORDER_TYPE_LIMIT and price > 0:
            order_params["price"] = price

        order_id = kite.place_order(**order_params)
        logger.info(f"Order placed successfully. Order ID: {order_id}")
        # Store pending order for status tracking (for live)
        EXECUTED_ORDERS[order_id] = {
            'instrument_token': instrument_token,
            'tradingsymbol': tradingsymbol,
            'quantity': quantity,
            'type': transaction_type,
            'status': 'pending' # Will update to 'complete' once confirmed
        }
        return order_id
    except Exception as e:
        logger.error(f"Error placing order for {tradingsymbol}: {e}")
        return None

def update_active_positions(tradingsymbol, transaction_type, quantity, price):
    if transaction_type == kite.TRANSACTION_TYPE_BUY:
        # Calculate new average entry price
        current_total_value = ACTIVE_POSITIONS[tradingsymbol]['quantity'] * ACTIVE_POSITIONS[tradingsymbol]['entry_price']
        new_total_value = current_total_value + (quantity * price)
        ACTIVE_POSITIONS[tradingsymbol]['quantity'] += quantity
        ACTIVE_POSITIONS[tradingsymbol]['entry_price'] = new_total_value / ACTIVE_POSITIONS[tradingsymbol]['quantity'] if ACTIVE_POSITIONS[tradingsymbol]['quantity'] > 0 else 0
    elif transaction_type == kite.TRANSACTION_TYPE_SELL:
        ACTIVE_POSITIONS[tradingsymbol]['quantity'] -= quantity
        # If position is fully squared off or reversed, adjust entry price logic
        if ACTIVE_POSITIONS[tradingsymbol]['quantity'] == 0:
            ACTIVE_POSITIONS[tradingsymbol]['entry_price'] = 0
            # Trigger P&L calculation for realized P&L when position is closed
            calculate_realized_pnl({'tradingsymbol': tradingsymbol, 'quantity': quantity, 'exit_price': price, 'entry_price': ACTIVE_POSITIONS[tradingsymbol]['entry_price']})

    logger.debug(f"Updated position for {tradingsymbol}: {ACTIVE_POSITIONS[tradingsymbol]}")

def get_ltp(instrument_token):
    global kite
    if PAPER_TRADING_MODE:
        # For paper trading, return a random price or mock data
        # In a real paper trading setup, you'd feed real-time data
        return 24000 + (math.sin(time.time() / 10) * 100) # Mock LTP

    if not kite:
        logger.error("Kite object not initialized. Cannot fetch LTP.")
        return None
    try:
        data = kite.ltp(f"NFO:{instrument_token}") # Assuming NFO segment
        return data[f"NFO:{instrument_token}"]['last_price']
    except Exception as e:
        logger.error(f"Error fetching LTP for {instrument_token}: {e}")
        # Attempt to reconnect on API error
        reconnect_kite(userdata.get('KITE_API_KEY'), userdata.get('KITE_API_SECRET'), userdata.get('KITE_REQUEST_TOKEN'))
        return None

def get_historical_data(instrument_token, from_date, to_date, interval):
    global kite
    if PAPER_TRADING_MODE:
        # Return mock historical data for paper trading
        logger.info("[PAPER TRADE] Returning mock historical data.")
        df = pd.DataFrame({
            'date': pd.to_datetime([from_date + datetime.timedelta(minutes=3*i) for i in range(10)]),
            'open': [23900 + i for i in range(10)],
            'high': [23910 + i for i in range(10)],
            'low': [23890 + i for i in range(10)],
            'close': [23905 + i for i in range(10)],
            'volume': [1000 + i*100 for i in range(10)]
        })
        return df.to_dict('records')

    if not kite:
        logger.error("Kite object not initialized. Cannot fetch historical data.")
        return None
    try:
        data = kite.historical_data(instrument_token, from_date, to_date, interval, continuous=False)
        return data
    except Exception as e:
        logger.error(f"Error fetching historical data for {instrument_token}: {e}")
        reconnect_kite(userdata.get('KITE_API_KEY'), userdata.get('KITE_API_SECRET'), userdata.get('KITE_REQUEST_TOKEN'))
        return None

def get_option_chain(nifty_spot_price, expiry_date=None):
    # This is a simplified mock. In reality, you'd use an API to fetch the full option chain
    # For Zerodha, you might need to iterate through all NFO instruments or use a third-party data provider.
    # For this example, we will generate theoretical strikes around the spot price.

    nearest_atm_strike = round(nifty_spot_price / 50) * 50
    strikes = sorted(list(set([
        nearest_atm_strike - 200, nearest_atm_strike - 150, nearest_atm_strike - 100, nearest_atm_strike - 50,
        nearest_atm_strike,
        nearest_atm_strike + 50, nearest_atm_strike + 100, nearest_atm_strike + 150, nearest_atm_strike + 200
    ])))

    options_data = []
    # Generate mock instrument tokens and symbols for Nifty options (CE and PE)
    # In live, you would fetch these from kite.instruments('NFO')
    mock_token_counter = 100000
    for strike in strikes:
        # Mock CE
        ce_symbol = f"NIFTY{expiry_date.strftime('%y%b%d').upper()}C{strike}"
        options_data.append({
            'instrument_token': mock_token_counter,
            'tradingsymbol': ce_symbol,
            'name': 'NIFTY',
            'expiry': expiry_date,
            'strike': strike,
            'instrument_type': 'CE',
            'segment': 'NFO'
        })
        mock_token_counter += 1

        # Mock PE
        pe_symbol = f"NIFTY{expiry_date.strftime('%y%b%d').upper()}P{strike}"
        options_data.append({
            'instrument_token': mock_token_counter,
            'tradingsymbol': pe_symbol,
            'name': 'NIFTY',
            'expiry': expiry_date,
            'strike': strike,
            'instrument_type': 'PE',
            'segment': 'NFO'
        })
        mock_token_counter += 1

    df_options = pd.DataFrame(options_data)
    logger.info(f"Generated mock option chain for spot {nifty_spot_price}.")
    return df_options

def get_nearest_expiry():
    today = datetime.date.today()
    # Nifty options typically expire on Thursdays
    # Find the next Thursday
    days_until_thursday = (3 - today.weekday() + 7) % 7 # 3 is Thursday
    if days_until_thursday == 0 and today.weekday() == 3: # If today is Thursday
        # Check if market is still open, if past market close, use next week's expiry
        if datetime.datetime.now().time() > MARKET_CLOSE_TIME:
             next_thursday = today + datetime.timedelta(days=7)
        else:
             next_thursday = today # Use today's expiry if before market close
    elif days_until_thursday == 0: # If today is Thursday but earlier in the day
        next_thursday = today
    else:
        next_thursday = today + datetime.timedelta(days=days_until_thursday)

    logger.info(f"Nearest expiry date: {next_thursday.strftime('%Y-%m-%d')}")
    return next_thursday

# --- Emergency Kill Switch ---
EMERGENCY_KILL_SWITCH_ACTIVE = False # Set to True to immediately square off all positions and stop the bot

def activate_kill_switch():
    global EMERGENCY_KILL_SWITCH_ACTIVE
    EMERGENCY_KILL_SWITCH_ACTIVE = True
    logger.warning("EMERGENCY KILL SWITCH ACTIVATED! All positions will be squared off.")

def deactivate_kill_switch():
    global EMERGENCY_KILL_SWITCH_ACTIVE
    EMERGENCY_KILL_SWITCH_ACTIVE = False
    logger.info("EMERGENCY KILL SWITCH DEACTIVATED.")

# Example usage for manual intervention via dashboard (concept)
# In a real dashboard, a button click would call activate_kill_switch()
# For this Colab script, you'd manually set EMERGENCY_KILL_SWITCH_ACTIVE = True
# in a cell above or within the main loop for testing.



### 3. Strategy Logic (Entry, Stop Loss, Re-entry, Trailing P&L)

This section defines the core trading logic, including how positions are initiated, managed, and closed based on market conditions.

In [ ]:
from google.colab import userdata

# Global variables for strategy state
first_candle_high = 0
first_candle_low = 0
first_candle_close = 0
initial_nifty_spot_price = 0

TRADE_INITIATED = False
SPREAD_TYPE = None # 'CALL_SPREAD' or 'PUT_SPREAD'

# Stores details of the currently active spread positions {symbol: {data}}
# This will store the buy leg and sell leg details separately
ACTIVE_SPREAD_POSITIONS = {
    'buy_leg': None, # {'instrument_token': token, 'tradingsymbol': symbol, 'strike': strike, 'type': 'CE'/'PE', 'entry_price': price, 'quantity': qty}
    'sell_leg': None # {'instrument_token': token, 'tradingsymbol': symbol, 'strike': strike, 'type': 'CE'/'PE', 'entry_price': price, 'quantity': qty, 'stop_loss_price': price, 're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}}
}

CURRENT_LOCKED_PROFIT = 0 # Dynamic profit lock value

def get_nifty_spot_price():
    # In a live environment, you'd get the actual Nifty spot price.
    # Zerodha Kite does not directly provide 'Nifty Spot Index' as a tradable instrument via LTP.
    # You often need to get Nifty Futures LTP (e.g., NIFTY24MAYFUT) or use a third-party data feed.
    # For this simulation, we'll use a mock value, or assume NIFTY_FUTURES_INSTRUMENT_TOKEN gives a good proxy.
    if PAPER_TRADING_MODE:
        mock_spot = 23925 + (math.sin(time.time() / 60) * 50) # Oscillate around 23925
        logger.info(f"[PAPER TRADE] Mock Nifty Spot Price: {mock_spot:.2f}")
        return mock_spot

    # For live, fetch LTP of Nifty Futures as a proxy for spot
    spot_ltp = get_ltp(NIFTY_FUTURES_INSTRUMENT_TOKEN)
    if spot_ltp:
        logger.info(f"Nifty Futures (Spot Proxy) LTP: {spot_ltp:.2f}")
        return spot_ltp
    logger.error("Could not fetch Nifty spot price.")
    return None

def get_atm_strike(spot_price):
    # Round to the nearest 50. If spot is 23925, ATM is 23950. If 23924, ATM is 23900.
    return round(spot_price / 50) * 50

def select_strikes(nifty_spot_price, option_type): # option_type: 'CE' or 'PE'
    atm_strike = get_atm_strike(nifty_spot_price)
    nearest_expiry = get_nearest_expiry()
    option_chain_df = get_option_chain(nifty_spot_price, nearest_expiry)

    buy_strike = 0
    sell_strike = 0

    if option_type == 'PE': # Put Spread: Buy OTM, Sell ITM
        # Buy: ATM to 50 point OTM-1 strike (e.g., if ATM is 23950, buy 23900 PE)
        buy_strike = atm_strike - OPTION_STRIKE_DIFFERENCE
        # Sell: ATM to 50 point ITM-1 strike (e.g., if ATM is 23950, sell 24000 PE)
        sell_strike = atm_strike + OPTION_STRIKE_DIFFERENCE
    elif option_type == 'CE': # Call Spread: Buy OTM, Sell ITM
        # Buy: ATM to 50 point OTM-1 strike (e.g., if ATM is 23950, buy 24000 CE)
        buy_strike = atm_strike + OPTION_STRIKE_DIFFERENCE
        # Sell: ATM to 50 point ITM-1 strike (e.g., if ATM is 23950, sell 23900 CE)
        sell_strike = atm_strike - OPTION_STRIKE_DIFFERENCE

    # Find instrument tokens for the selected strikes
    buy_option = option_chain_df[(option_chain_df['strike'] == buy_strike) & (option_chain_df['instrument_type'] == option_type)].iloc[0] if not option_chain_df[(option_chain_df['strike'] == buy_strike) & (option_chain_df['instrument_type'] == option_type)].empty else None
    sell_option = option_chain_df[(option_chain_df['strike'] == sell_strike) & (option_chain_df['instrument_type'] == option_type)].iloc[0] if not option_chain_df[(option_chain_df['strike'] == sell_strike) & (option_chain_df['instrument_type'] == option_type)].empty else None

    if buy_option is None or sell_option is None:
        logger.error(f"Could not find options for selected strikes: Buy {buy_strike} {option_type}, Sell {sell_strike} {option_type}")
        return None, None

    logger.info(f"Selected strikes: Buy {buy_option['tradingsymbol']}, Sell {sell_option['tradingsymbol']}")
    return buy_option, sell_option

def enter_spread(spread_type, nifty_spot_price):
    global TRADE_INITIATED, SPREAD_TYPE, ACTIVE_SPREAD_POSITIONS
    if TRADE_INITIATED:
        logger.warning("Trade already initiated. Cannot enter new spread.")
        return False

    option_type = 'PE' if spread_type == 'PUT_SPREAD' else 'CE'
    buy_option, sell_option = select_strikes(nifty_spot_price, option_type)

    if buy_option is None or sell_option is None:
        logger.error("Failed to select appropriate strikes. Aborting spread entry.")
        return False

    # Place buy leg order
    buy_order_id = place_order(buy_option['instrument_token'], buy_option['tradingsymbol'], kite.TRANSACTION_TYPE_BUY, QUANTITY_PER_LOT)
    time.sleep(1) # Small delay between orders
    # Place sell leg order
    sell_order_id = place_order(sell_option['instrument_token'], sell_option['tradingsymbol'], kite.TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT)

    if buy_order_id and sell_order_id:
        # Assuming immediate execution for simplicity in paper trade. Live would need order status checks.
        ACTIVE_SPREAD_POSITIONS['buy_leg'] = {
            'instrument_token': buy_option['instrument_token'],
            'tradingsymbol': buy_option['tradingsymbol'],
            'strike': buy_option['strike'],
            'type': option_type,
            'entry_price': EXECUTED_ORDERS[buy_order_id].get('average_price'),
            'quantity': QUANTITY_PER_LOT,
            'order_id': buy_order_id
        }
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
            'instrument_token': sell_option['instrument_token'],
            'tradingsymbol': sell_option['tradingsymbol'],
            'strike': sell_option['strike'],
            'type': option_type,
            'entry_price': EXECUTED_ORDERS[sell_order_id].get('average_price'),
            'quantity': QUANTITY_PER_LOT,
            'order_id': sell_order_id
        }

        # Set 1st Stop Loss for selling leg
        if spread_type == 'PUT_SPREAD':
            ACTIVE_SPREAD_POSITIONS['sell_leg']['stop_loss_price'] = first_candle_low # Stop loss on Low of first 3-min candle close price
            logger.info(f"Put Spread entered. Sell Leg SL set to first_candle_low: {first_candle_low}")
        elif spread_type == 'CALL_SPREAD':
            ACTIVE_SPREAD_POSITIONS['sell_leg']['stop_loss_price'] = first_candle_high # Stop loss on High of first 3-min candle close price
            logger.info(f"Call Spread entered. Sell Leg SL set to first_candle_high: {first_candle_high}")

        TRADE_INITIATED = True
        SPREAD_TYPE = spread_type
        logger.info(f"Successfully entered {spread_type}.")
        return True
    else:
        logger.error(f"Failed to enter {spread_type}. Rolling back any partially placed orders (not implemented).")
        # TODO: Add logic to square off partially placed orders if one leg fails
        return False

def get_net_pnl():
    global NET_REALIZED_PNL, NET_UNREALIZED_PNL

    current_pnl = NET_REALIZED_PNL # Start with realized P&L

    # Calculate unrealized P&L for active buy_leg
    if ACTIVE_SPREAD_POSITIONS['buy_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['buy_leg']
        ltp = get_ltp(leg['instrument_token'])
        if ltp is not None and leg['entry_price']:
            current_pnl += (ltp - leg['entry_price']) * leg['quantity']

    # Calculate unrealized P&L for active sell_leg
    if ACTIVE_SPREAD_POSITIONS['sell_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        ltp = get_ltp(leg['instrument_token'])
        if ltp is not None and leg['entry_price']:
            current_pnl += (leg['entry_price'] - ltp) * leg['quantity'] # For sell leg, entry - current_price

    NET_UNREALIZED_PNL = current_pnl - NET_REALIZED_PNL # Update global unrealized PNL (conceptual)
    logger.debug(f"Current Net P&L: {current_pnl:.2f}")
    return current_pnl

def update_dynamic_profit_lock(current_pnl):
    global CURRENT_LOCKED_PROFIT
    if not DYNAMIC_PROFIT_LOCK_ENABLED:
        return

    if current_pnl >= INITIAL_PROFIT_LOCK_TRIGGER:
        # Ensure locked_profit never decreases and maintains the buffer
        new_locked_profit = current_pnl - PROFIT_LOCK_BUFFER
        if new_locked_profit > CURRENT_LOCKED_PROFIT:
            CURRENT_LOCKED_PROFIT = new_locked_profit
            logger.info(f"Dynamic Profit Lock updated. New locked_profit: {CURRENT_LOCKED_PROFIT:.2f}")

def square_off_all_positions(reason="Bot triggered square-off"):
    global TRADE_INITIATED, ACTIVE_SPREAD_POSITIONS
    logger.warning(f"Squaring off all active positions. Reason: {reason}")

    if ACTIVE_SPREAD_POSITIONS['buy_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['buy_leg']
        logger.info(f"Squaring off buy leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], kite.TRANSACTION_TYPE_SELL, leg['quantity'])
        ACTIVE_SPREAD_POSITIONS['buy_leg'] = None

    if ACTIVE_SPREAD_POSITIONS['sell_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        logger.info(f"Squaring off sell leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], kite.TRANSACTION_TYPE_BUY, leg['quantity'])
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = None

    TRADE_INITIATED = False
    logger.info("All positions squared off.")

def check_daily_max_loss():
    # This requires tracking actual day's P&L, which is more complex than just current_pnl
    # For now, we'll use current_pnl for demonstration, but in live, it would be sum of realized P&L + current unrealized P&L
    current_net_pnl = get_net_pnl() # This is a positive value for profit, negative for loss
    if current_net_pnl < -DAILY_MAX_LOSS:
        logger.critical(f"DAILY MAX LOSS REACHED! Current P&L: {current_net_pnl:.2f}. Limit: {-DAILY_MAX_LOSS:.2f}")
        square_off_all_positions(reason="Daily Max Loss reached.")
        return True # Indicate that max loss was hit and bot should stop
    return False

def check_and_handle_sell_leg_sl(current_nifty_close, current_candle_high, current_candle_low):
    global ACTIVE_SPREAD_POSITIONS
    sell_leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
    if not sell_leg or 'stop_loss_price' not in sell_leg:
        return # No active sell leg or SL not set

    sl_triggered = False
    if SPREAD_TYPE == 'CALL_SPREAD': # SL = first_candle_high, re-entry on red candle low break
        # If any candle gives closing confirmation on the High of the first 3-min candle Closing price
        # For call spread, we short CE. Price going up is bad. So SL is hit if current Nifty goes above first_candle_high
        # This is a bit ambiguous: 'closing confirmation on the High of the first 3-min candle Closing price'
        # I'll interpret this as: if current_nifty_close > first_candle_high for a Call Spread.
        if current_nifty_close > sell_leg['stop_loss_price']:
            logger.warning(f"Call Spread Sell Leg SL Triggered! Nifty Close ({current_nifty_close:.2f}) > SL Price ({sell_leg['stop_loss_price']:.2f})")
            sl_triggered = True

    elif SPREAD_TYPE == 'PUT_SPREAD': # SL = first_candle_low, re-entry on green candle high break
        # If any candle gives closing confirmation on the Low of the first 3-min candle Closing price
        # For put spread, we short PE. Price going down is bad. So SL is hit if current Nifty goes below first_candle_low
        # I'll interpret this as: if current_nifty_close < first_candle_low for a Put Spread.
        if current_nifty_close < sell_leg['stop_loss_price']:
            logger.warning(f"Put Spread Sell Leg SL Triggered! Nifty Close ({current_nifty_close:.2f}) < SL Price ({sell_leg['stop_loss_price']:.2f})")
            sl_triggered = True

    if sl_triggered:
        # Square off the sell leg
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        logger.info(f"Squaring off SL triggered sell leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], kite.TRANSACTION_TYPE_BUY, leg['quantity'], reason="Sell Leg SL Hit")
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = None # Remove sell leg position

        logger.info("Initiating re-entry sequence for sell leg...")
        # Reset re_entry_mark for next re-entry attempt
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {'instrument_token': None, 'tradingsymbol': None, 'strike': None, 'type': None, 'entry_price': None, 'quantity': 0, 'stop_loss_price': None, 're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}}
        return True # SL was hit
    return False # SL not hit

def attempt_sell_leg_reentry(current_nifty_close, current_candle_open, current_candle_high, current_candle_low):
    global ACTIVE_SPREAD_POSITIONS
    sell_leg_state = ACTIVE_SPREAD_POSITIONS['sell_leg']
    if not sell_leg_state or sell_leg_state['quantity'] > 0: # Only re-enter if sell leg is not active (i.e., squared off)
        return False

    re_entry_mark = sell_leg_state['re_entry_mark']

    if SPREAD_TYPE == 'CALL_SPREAD': # Re-entry rules after 1st SL for Call Spread (short CE)
        # Wait for a red candle (bearish candle). Mark its Low & High.
        # Re-sell the same option strike if next candle closes below the red candle’s Low.
        # If price closes above the red candle’s High -> discard that trigger and wait for next red candle.
        # New Stop loss after Re-entry = High of the Confirmation red candle.
        if re_entry_mark['candle_type'] is None: # Looking for first red candle
            if current_nifty_close < current_candle_open: # Found a red candle
                re_entry_mark['low'] = current_candle_low
                re_entry_mark['high'] = current_candle_high
                re_entry_mark['candle_type'] = 'RED'
                logger.info(f"[CALL SPREAD RE-ENTRY] Marked first red candle: Low={current_candle_low}, High={current_candle_high}")
        elif re_entry_mark['candle_type'] == 'RED': # Red candle marked, looking for confirmation
            if current_nifty_close < re_entry_mark['low']: # Confirmation: next candle closes below red candle's Low
                logger.info(f"[CALL SPREAD RE-ENTRY] Re-entry confirmed! Nifty Close ({current_nifty_close:.2f}) < Red Candle Low ({re_entry_mark['low']:.2f})")
                # Re-sell the same option strike (assuming it's still available and valid)
                # Need to find the instrument token for the original sell_leg strike again
                original_sell_strike = ACTIVE_SPREAD_POSITIONS['buy_leg']['strike'] - OPTION_STRIKE_DIFFERENCE # Reverse calc
                option_type = 'CE'
                _, sell_option_for_reentry = select_strikes(current_nifty_close, option_type) # Use current nifty close to get instrument info

                if sell_option_for_reentry:
                    re_entry_order_id = place_order(sell_option_for_reentry['instrument_token'], sell_option_for_reentry['tradingsymbol'], kite.TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT, reason="Call Spread Re-entry")
                    if re_entry_order_id:
                        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
                            'instrument_token': sell_option_for_reentry['instrument_token'],
                            'tradingsymbol': sell_option_for_reentry['tradingsymbol'],
                            'strike': sell_option_for_reentry['strike'],
                            'type': option_type,
                            'entry_price': EXECUTED_ORDERS[re_entry_order_id].get('average_price'),
                            'quantity': QUANTITY_PER_LOT,
                            'order_id': re_entry_order_id,
                            'stop_loss_price': re_entry_mark['high'], # New SL is High of Confirmation red candle
                            're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None} # Reset for next potential re-entry
                        }
                        logger.info(f"[CALL SPREAD RE-ENTRY] Re-entered Call Spread Sell Leg. New SL: {re_entry_mark['high']:.2f}")
                        return True
                logger.error("[CALL SPREAD RE-ENTRY] Failed to find or place re-entry order.")
                # Discard trigger if re-entry fails
                re_entry_mark['candle_type'] = None

            elif current_nifty_close > re_entry_mark['high']: # Discard trigger
                logger.info(f"[CALL SPREAD RE-ENTRY] Discarded re-entry trigger. Nifty Close ({current_nifty_close:.2f}) > Red Candle High ({re_entry_mark['high']:.2f})")
                re_entry_mark['candle_type'] = None # Wait for next red candle

    elif SPREAD_TYPE == 'PUT_SPREAD': # Re-entry rules after 1st SL for Put Spread (short PE)
        # Wait for a green candle (bullish candle). Mark its Low & High.
        # Re-sell the same option strike if next candle closes above the green candle’s High.
        # If price closes below the green candle’s Low -> discard that trigger and wait for next green candle.
        # New Stop loss after Re-entry = Low of the Confirmation green candle.
        if re_entry_mark['candle_type'] is None: # Looking for first green candle
            if current_nifty_close > current_candle_open: # Found a green candle
                re_entry_mark['low'] = current_candle_low
                re_entry_mark['high'] = current_candle_high
                re_entry_mark['candle_type'] = 'GREEN'
                logger.info(f"[PUT SPREAD RE-ENTRY] Marked first green candle: Low={current_candle_low}, High={current_candle_high}")
        elif re_entry_mark['candle_type'] == 'GREEN': # Green candle marked, looking for confirmation
            if current_nifty_close > re_entry_mark['high']: # Confirmation: next candle closes above green candle's High
                logger.info(f"[PUT SPREAD RE-ENTRY] Re-entry confirmed! Nifty Close ({current_nifty_close:.2f}) > Green Candle High ({re_entry_mark['high']:.2f})")
                # Re-sell the same option strike
                original_sell_strike = ACTIVE_SPREAD_POSITIONS['buy_leg']['strike'] + OPTION_STRIKE_DIFFERENCE # Reverse calc
                option_type = 'PE'
                _, sell_option_for_reentry = select_strikes(current_nifty_close, option_type)

                if sell_option_for_reentry:
                    re_entry_order_id = place_order(sell_option_for_reentry['instrument_token'], sell_option_for_reentry['tradingsymbol'], kite.TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT, reason="Put Spread Re-entry")
                    if re_entry_order_id:
                        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
                            'instrument_token': sell_option_for_reentry['instrument_token'],
                            'tradingsymbol': sell_option_for_reentry['tradingsymbol'],
                            'strike': sell_option_for_reentry['strike'],
                            'type': option_type,
                            'entry_price': EXECUTED_ORDERS[re_entry_order_id].get('average_price'),
                            'quantity': QUANTITY_PER_LOT,
                            'order_id': re_entry_order_id,
                            'stop_loss_price': re_entry_mark['low'], # New SL is Low of Confirmation green candle
                            're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None} # Reset for next potential re-entry
                        }
                        logger.info(f"[PUT SPREAD RE-ENTRY] Re-entered Put Spread Sell Leg. New SL: {re_entry_mark['low']:.2f}")
                        return True
                logger.error("[PUT SPREAD RE-ENTRY] Failed to find or place re-entry order.")
                # Discard trigger if re-entry fails
                re_entry_mark['candle_type'] = None

            elif current_nifty_close < re_entry_mark['low']: # Discard trigger
                logger.info(f"[PUT SPREAD RE-ENTRY] Discarded re-entry trigger. Nifty Close ({current_nifty_close:.2f}) < Green Candle Low ({re_entry_mark['low']:.2f})")
                re_entry_mark['candle_type'] = None # Wait for next green candle

    return False

def main_strategy_loop():
    global first_candle_high, first_candle_low, first_candle_close, initial_nifty_spot_price, TRADE_INITIATED, CURRENT_LOCKED_PROFIT, EMERGENCY_KILL_SWITCH_ACTIVE

    logger.info("Starting Nifty 50 Options Trading Bot.")

    # Connect to Kite (requires manual step to get request_token if not paper trading)
    # For paper trading, you might just initialize without a token if mocking all calls.
    # In a real setup, a Flask app or similar would handle the OAuth callback for request_token.
    if not PAPER_TRADING_MODE:
        # Access credentials from Colab secrets
        KITE_API_KEY = userdata.get('KITE_API_KEY')
        KITE_API_SECRET = userdata.get('KITE_API_SECRET')
        KITE_REQUEST_TOKEN = userdata.get('KITE_REQUEST_TOKEN', default=None) # Might be None initially
        if KITE_REQUEST_TOKEN is None:
             logger.error("KITE_REQUEST_TOKEN not found. Please generate it via Zerodha login_url and store in secrets.")
             print(f"Login URL: {KiteConnect(api_key=KITE_API_KEY).login_url()}")
             return

        global kite
        kite = initialize_kite_api(KITE_API_KEY, KITE_API_SECRET, KITE_REQUEST_TOKEN)
        if not kite:
            logger.critical("Failed to initialize Kite API. Exiting bot.")
            return
    else:
        logger.info("Running in PAPER TRADING MODE. Mocking Kite API calls.")

    # --- Main Trading Loop ---
    while True:
        current_time = datetime.datetime.now().time()
        today_date = datetime.date.today()
        logger.debug(f"Current time: {current_time}")

        if EMERGENCY_KILL_SWITCH_ACTIVE:
            square_off_all_positions(reason="Emergency Kill Switch Activated")
            logger.critical("Emergency Kill Switch is active. Stopping bot.")
            break # Exit the loop

        # Check daily max loss at the beginning of each cycle
        if check_daily_max_loss():
            logger.critical("Daily max loss reached. Stopping strategy for the day.")
            break # Exit the loop for the day

        # --- Phase 1: Determine First 3-min Candle Close ---
        if current_time < FIRST_CANDLE_END_TIME and not first_candle_close:
            # Fetch 3-min historical data for Nifty Futures (proxy for spot)
            from_ts = datetime.datetime.combine(today_date, FIRST_CANDLE_START_TIME)
            to_ts = datetime.datetime.combine(today_date, FIRST_CANDLE_END_TIME)
            # Adjust for API call, usually current time - some buffer for `to_date` if candle isn't formed yet.
            # For exact 9:15-9:18 candle, we need to wait until 9:18:01 to request it.
            if current_time >= FIRST_CANDLE_END_TIME:
                # This block will be executed once at or after 9:18
                try:
                    historical_data = get_historical_data(NIFTY_FUTURES_INSTRUMENT_TOKEN, from_ts, to_ts, "3minute")
                    if historical_data and len(historical_data) > 0:
                        first_candle_data = historical_data[-1] # Assuming last candle is the 9:15-9:18 one
                        first_candle_high = first_candle_data['high']
                        first_candle_low = first_candle_data['low']
                        first_candle_close = first_candle_data['close']
                        initial_nifty_spot_price = historical_data[0]['open'] # Use 9:15 open as reference for percentage calc
                        logger.info(f"First 3-min candle (9:15-9:18) captured: High={first_candle_high:.2f}, Low={first_candle_low:.2f}, Close={first_candle_close:.2f}")
                    else:
                        logger.warning("No historical data for first 3-min candle yet or data not available.")
                except Exception as e:
                    logger.error(f"Error fetching first 3-min candle data: {e}")

        # --- Phase 2: Entry Conditions (after first 3-min candle is formed) ---
        if first_candle_close and not TRADE_INITIATED and current_time > FIRST_CANDLE_END_TIME:
            current_nifty_spot = get_nifty_spot_price()
            if current_nifty_spot is None:
                time.sleep(5)
                continue

            percentage_change = ((current_nifty_spot - initial_nifty_spot_price) / initial_nifty_spot_price) * 100
            logger.info(f"Current Nifty Spot: {current_nifty_spot:.2f}, Initial Nifty Spot: {initial_nifty_spot_price:.2f}, Change: {percentage_change:.2f}%")

            if percentage_change > PROFIT_ENTRY_THRESHOLD_PERCENT:
                logger.info(f"Nifty Close > +{PROFIT_ENTRY_THRESHOLD_PERCENT}% (Actual: {percentage_change:.2f}%). Entering Put Spread.")
                enter_spread('PUT_SPREAD', current_nifty_spot)
            elif percentage_change < -PROFIT_ENTRY_THRESHOLD_PERCENT:
                logger.info(f"Nifty Close < -{PROFIT_ENTRY_THRESHOLD_PERCENT}% (Actual: {percentage_change:.2f}%). Entering Call Spread.")
                enter_spread('CALL_SPREAD', current_nifty_spot)

        # --- Phase 3: Manage Active Trades (Stop Loss, Re-entry, Dynamic Profit Lock) ---
        if TRADE_INITIATED:
            current_nifty_spot = get_nifty_spot_price()
            if current_nifty_spot is None:
                time.sleep(5)
                continue

            # For SL and re-entry, we need current candle info. Let's fetch 3-min candle data continuously.
            # This is a simplification; in real-time, you'd use WebSocket for tick data or 1-min candles.
            last_candle_data = None
            try:
                # Fetch the most recent completed 3-min candle
                history_to_time = datetime.datetime.now()
                history_from_time = history_to_time - datetime.timedelta(minutes=5) # Look back a bit
                recent_candles = get_historical_data(NIFTY_FUTURES_INSTRUMENT_TOKEN, history_from_time, history_to_time, "3minute")
                if recent_candles and len(recent_candles) > 0:
                    last_candle_data = recent_candles[-1] # Most recent completed candle
                    current_candle_high = last_candle_data['high']
                    current_candle_low = last_candle_data['low']
                    current_candle_open = last_candle_data['open']
                    current_nifty_close_for_sl = last_candle_data['close']
                    logger.debug(f"Latest 3-min candle: O={current_candle_open}, H={current_candle_high}, L={current_candle_low}, C={current_nifty_close_for_sl}")
                else:
                    logger.warning("No recent 3-min candle data available for SL/Re-entry check.")
            except Exception as e:
                logger.error(f"Error fetching recent candle for SL/Re-entry: {e}")
                last_candle_data = None # Ensure it's None if error

            if ACTIVE_SPREAD_POSITIONS['sell_leg'] and ACTIVE_SPREAD_POSITIONS['sell_leg']['quantity'] > 0: # If sell leg is active
                if last_candle_data: # Only check SL if we have recent candle data
                    sl_hit = check_and_handle_sell_leg_sl(current_nifty_close_for_sl, current_candle_high, current_candle_low)
                    if sl_hit: # If SL was hit, re-entry logic will be attempted in next loop iteration or immediately if more data available
                        pass # Proceed to check for re-entry opportunities
            else: # If sell leg is NOT active (i.e., it was squared off and needs re-entry)
                if last_candle_data: # Only attempt re-entry if we have recent candle data
                    attempt_sell_leg_reentry(current_nifty_close_for_sl, current_candle_open, current_candle_high, current_candle_low)

            # Dynamic Profit Lock Check
            current_net_pnl = get_net_pnl()
            update_dynamic_profit_lock(current_net_pnl)

            if DYNAMIC_PROFIT_LOCK_ENABLED and CURRENT_LOCKED_PROFIT > 0:
                if current_net_pnl <= CURRENT_LOCKED_PROFIT:
                    logger.warning(f"Dynamic Profit Lock Triggered! Current P&L ({current_net_pnl:.2f}) <= Locked Profit ({CURRENT_LOCKED_PROFIT:.2f})")
                    square_off_all_positions(reason="Dynamic Profit Lock triggered.")
                    CURRENT_LOCKED_PROFIT = 0 # Reset locked profit after square-off

        # --- Phase 4: Auto Square-off at Market Close ---
        if current_time >= MARKET_CLOSE_TIME and TRADE_INITIATED:
            logger.info(f"Market close time ({MARKET_CLOSE_TIME}) reached. Squaring off all positions.")
            square_off_all_positions(reason="Market close auto square-off.")
            # After square-off at close, stop for the day
            break

        # Sleep for a short interval before next iteration
        time.sleep(30) # Check every 30 seconds for new candle/price action

    logger.info("Nifty 50 Options Trading Bot stopped for the day.")

# To run the bot, you would call main_strategy_loop()
# main_strategy_loop()


### 5. Dashboard

In [19]:
import IPython.display as display

def render_dashboard():
    html_output = f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; border: 1px solid #ccc; padding: 15px; border-radius: 8px; background-color: #f9f9f9;">
        <h2 style="color: #333; margin-top: 0;">Algo Trading Dashboard</h2>
        <p style="font-size: 1.1em; color: #555;">Last updated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

        <div style="display: flex; justify-content: space-around; margin-bottom: 20px; border-bottom: 1px solid #eee; padding-bottom: 15px;">
            <div style="text-align: center;">
                <h3 style="color: #007bff;">P&L Overview</h3>
                <p style="margin: 5px 0;">Realized P&L: <strong style="color: {('green' if NET_REALIZED_PNL >= 0 else 'red')};">₹{NET_REALIZED_PNL:.2f}</strong></p>
                <p style="margin: 5px 0;">Unrealized P&L: <strong style="color: {('green' if NET_UNREALIZED_PNL >= 0 else 'red')};">₹{NET_UNREALIZED_PNL:.2f}</strong></p>
                <p style="margin: 5px 0;"><strong>Total P&L: <span style="color: {('green' if (NET_REALIZED_PNL + NET_UNREALIZED_PNL) >= 0 else 'red')}; font-size: 1.2em;">₹{(NET_REALIZED_PNL + NET_UNREALIZED_PNL):.2f}</span></strong></p>
            </div>
            <div style="text-align: center;">
                <h3 style="color: #007bff;">Risk Management</h3>
                <p style="margin: 5px 0;">Daily Max Loss: <strong style="color: red;">₹{DAILY_MAX_LOSS:.2f}</strong></p>
                <p style="margin: 5px 0;">Kill Switch: <strong style="color: {('red' if EMERGENCY_KILL_SWITCH_ACTIVE else 'green')};">{'ACTIVE' if EMERGENCY_KILL_SWITCH_ACTIVE else 'INACTIVE'}</strong></p>
                <p style="margin: 5px 0;">Dynamic Profit Lock: <strong style="color: {('green' if DYNAMIC_PROFIT_LOCK_ENABLED else 'gray')};">{'ENABLED' if DYNAMIC_PROFIT_LOCK_ENABLED else 'DISABLED'}</strong> (Locked: ₹{CURRENT_LOCKED_PROFIT:.2f})</p>
            </div>
            <div style="text-align: center;">
                <h3 style="color: #007bff;">Strategy State</h3>
                <p style="margin: 5px 0;">Trading Mode: <strong style="color: {'orange' if PAPER_TRADING_MODE else 'purple'};">{'PAPER' if PAPER_TRADING_MODE else 'LIVE'}</strong></p>
                <p style="margin: 5px 0;">Trade Initiated: <strong style="color: {('green' if TRADE_INITIATED else 'gray')};">{'YES' if TRADE_INITIATED else 'NO'}</strong></p>
                <p style="margin: 5px 0;">Spread Type: <strong>{SPREAD_TYPE if SPREAD_TYPE else 'N/A'}</strong></p>
            </div>
        </div>

        <h3 style="color: #007bff;">Active Positions</h3>
        <table style="width: 100%; border-collapse: collapse; margin-bottom: 20px;">
            <thead>
                <tr style="background-color: #e9ecef;">
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Leg</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Symbol</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Quantity</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Entry Price</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Current LTP</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">P&L</th>
                </tr>
            </thead>
            <tbody>
    """
    if ACTIVE_SPREAD_POSITIONS['buy_leg']:
        buy_leg = ACTIVE_SPREAD_POSITIONS['buy_leg']
        buy_ltp = get_ltp(buy_leg['instrument_token']) if buy_leg['instrument_token'] else 0
        buy_pnl = (buy_ltp - buy_leg['entry_price']) * buy_leg['quantity'] if buy_ltp and buy_leg['entry_price'] else 0
        html_output += f"""
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd;">Buy Leg</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{buy_leg['tradingsymbol']}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{buy_leg['quantity']}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">₹{buy_leg['entry_price']:.2f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">₹{buy_ltp:.2f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: {('green' if buy_pnl >= 0 else 'red')};">₹{buy_pnl:.2f}</td>
                </tr>
        """
    if ACTIVE_SPREAD_POSITIONS['sell_leg']:
        sell_leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        sell_ltp = get_ltp(sell_leg['instrument_token']) if sell_leg['instrument_token'] else 0
        sell_pnl = (sell_leg['entry_price'] - sell_ltp) * sell_leg['quantity'] if sell_ltp and sell_leg['entry_price'] else 0
        html_output += f"""
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd;">Sell Leg</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{sell_leg['tradingsymbol']}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{sell_leg['quantity']}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">₹{sell_leg['entry_price']:.2f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">₹{sell_ltp:.2f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: {('green' if sell_pnl >= 0 else 'red')};">₹{sell_pnl:.2f}</td>
                </tr>
        """
    if not ACTIVE_SPREAD_POSITIONS['buy_leg'] and not ACTIVE_SPREAD_POSITIONS['sell_leg']:
        html_output += f"""
                <tr><td colspan="6" style="padding: 8px; border: 1px solid #ddd; text-align: center;">No Active Positions</td></tr>
        """

    html_output += f"""
            </tbody>
        </table>

        <h3 style="color: #007bff;">Executed Orders (Last 5)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-bottom: 20px;">
            <thead>
                <tr style="background-color: #e9ecef;">
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Order ID</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Symbol</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Type</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Qty</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Avg. Price</th>
                    <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Status</th>
                </tr>
            </thead>
            <tbody>
    """
    recent_orders = list(EXECUTED_ORDERS.items())[-5:] # Get last 5 orders
    if recent_orders:
        for order_id, details in reversed(recent_orders):
            html_output += f"""
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd; font-size: 0.9em;">{order_id}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{details.get('tradingsymbol', 'N/A')}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{details.get('type', 'N/A')}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{details.get('quantity', 'N/A')}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">₹{details.get('average_price', 0):.2f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{details.get('status', 'N/A')}</td>
                </tr>
            """
    else:
        html_output += f"""
                <tr><td colspan="6" style="padding: 8px; border: 1px solid #ddd; text-align: center;">No Orders Executed Yet</td></tr>
        """
    html_output += f"""
            </tbody>
        </table>

        <h3 style="color: #007bff;">Latest Logs</h3>
        <div style="background-color: #e0e0e0; padding: 10px; border-radius: 5px; max-height: 200px; overflow-y: auto; font-size: 0.85em; white-space: pre-wrap; word-break: break-word;">
    """
    try:
        with open(LOG_FILE, 'r') as f:
            log_lines = f.readlines()
            for line in log_lines[-10:]: # Show last 10 log lines
                html_output += f"<p style=\"margin: 2px 0;\">{line.strip()}</p>\n"
    except FileNotFoundError:
        html_output += "<p>Log file not found.</p>\n"
    except Exception as e:
        html_output += f"<p>Error reading log file: {e}</p>\n"

    html_output += f"""
        </div>
    </div>
    """
    display.clear_output(wait=True)
    display.display(display.HTML(html_output))



In [ ]:
# Call this function to display the dashboard at any time.
# If you run the main_strategy_loop in a separate cell, you can run this cell
# in another tab or a separate execution to get periodic updates.
# Be aware that running this in a loop will block the cell execution.

# Example of a static display:
render_dashboard()

In [17]:
from google.colab import userdata
from kiteconnect import KiteConnect

# Replace with your actual redirect URL configured in your Kite Connect app
YOUR_REDIRECT_URL = "http://localhost:3000"

# Retrieve your API Key from Colab Secrets
KITE_API_KEY = userdata.get('KITE_API_KEY')

if KITE_API_KEY:
    # Initialize KiteConnect with just the API key
    kite = KiteConnect(api_key=KITE_API_KEY)
    # Call login_url() with no arguments, assuming redirect_url is configured on Zerodha app settings
    login_url = kite.login_url()
    print(f"Please go to this URL to generate your request_token:\n{login_url}")
    print("After logging in, copy the 'request_token' from the redirected URL and save it in Colab Secrets as KITE_REQUEST_TOKEN.")
    print(f"Note: Ensure '{YOUR_REDIRECT_URL}' is configured as the redirect URL in your Zerodha Kite developer app settings.")
else:
    print("KITE_API_KEY not found in Colab secrets. Please add it first.")

Please go to this URL to generate your request_token:
https://kite.zerodha.com/connect/login?api_key=4b3rtj959l7p4r47&v=3
After logging in, copy the 'request_token' from the redirected URL and save it in Colab Secrets as KITE_REQUEST_TOKEN.
Note: Ensure 'http://localhost:3000' is configured as the redirect URL in your Zerodha Kite developer app settings.


### 4. Run the Trading Bot

Execute this cell to start the Nifty 50 Options Trading Bot. Monitor the logs for activity.

In [ ]:
# Start the main strategy loop
# Ensure you have configured your Kite API credentials in Colab Secrets and set PAPER_TRADING_MODE appropriately.
main_strategy_loop()


KeyboardInterrupt: 

In [ ]:
main_strategy_loop()

KeyboardInterrupt: 